In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

In [ ]:
current_directory = Path.cwd().parent
discharge_directory = Path.joinpath(current_directory, "Data Start Point")

points_of_interest_file_name = Path.joinpath(discharge_directory, "points_of_interest.json")
battery_state_file_name = Path.joinpath(discharge_directory, "battery_state.csv")
compiled_file_name = Path.joinpath(discharge_directory, "final_cleaned_compiled (with SoC).csv")

output_file_name = Path.joinpath(discharge_directory, "battery_constants.json")

In [3]:
compiled_df = pd.read_csv(compiled_file_name)
print(compiled_df.head())

   Translated Time  Translated Time (minutes)  \
0             1000                   0.016667   
1             1032                   0.017200   
2             1064                   0.017733   
3             1095                   0.018250   
4             1126                   0.018767   

   ADS Reading 1 (Smoothed and Offset)  Calibrated Current 1  \
0                             2.863429              0.008564   
1                            -1.136571             -0.003399   
2                             1.863429              0.005573   
3                             1.863429              0.005573   
4                            -3.136571             -0.009381   

   ADS Reading 2 (Smoothed and Offset)  Calibrated Current 2  \
0                             4.816303              0.014406   
1                             4.816303              0.014406   
2                             4.816303              0.014406   
3                             4.816303              0.014406   


In [4]:
sensor_draw_estimate_Ah = 0.081
sensor_draw_estimate_coulombs = sensor_draw_estimate_Ah * 3600

total_charge = compiled_df['Charge Change'].sum() - sensor_draw_estimate_coulombs
total_Ah = (total_charge / 3600)

print(f"Total Charge: {total_charge:.4f} Coulombs")
print(f"Total Ah: {total_Ah:.4f} Ah")

Total Charge: 264128.6780 Coulombs
Total Ah: 73.3691 Ah


In [5]:
points_of_interest = json.load(open(points_of_interest_file_name, 'r'))
before_close = points_of_interest['before_close']
before_open = points_of_interest['before_open']

In [6]:
STEP_BACK = 10
SAMPLING_RANGE = 50

sd_voltage_array = []

for index in before_close['index'][:-1]: # Exclude the last point which have the highest noise from shutting down
    end_index = max(0, index - STEP_BACK)
    start_index = max(0, end_index - SAMPLING_RANGE)
    mean_voltage = compiled_df.loc[start_index:end_index, 'Calibrated Voltage'].mean()
    sd_voltage = compiled_df.loc[start_index:end_index, 'Calibrated Voltage'].std()
    sd_voltage_array.append(sd_voltage)
    print(f"Mean Voltage: {mean_voltage:.4f} V, SD Voltage: {sd_voltage:.4f} V")

sigma_v = np.mean(sd_voltage_array)
print(f"Mean Standard Deviations: {sigma_v:.4f} V")

Mean Voltage: 58.2012 V, SD Voltage: 0.0014 V
Mean Voltage: 57.0507 V, SD Voltage: 0.0023 V
Mean Voltage: 56.3339 V, SD Voltage: 0.0020 V
Mean Voltage: 55.4058 V, SD Voltage: 0.0018 V
Mean Voltage: 54.1037 V, SD Voltage: 0.0021 V
Mean Voltage: 53.1237 V, SD Voltage: 0.0016 V
Mean Voltage: 52.3826 V, SD Voltage: 0.0019 V
Mean Voltage: 51.8017 V, SD Voltage: 0.0020 V
Mean Voltage: 51.3496 V, SD Voltage: 0.0016 V
Mean Voltage: 50.8410 V, SD Voltage: 0.0015 V
Mean Voltage: 50.2540 V, SD Voltage: 0.0019 V
Mean Voltage: 49.4435 V, SD Voltage: 0.0020 V
Mean Voltage: 49.1040 V, SD Voltage: 0.0021 V
Mean Standard Deviations: 0.0019 V


In [7]:
# During circuit open, where current = 0, almost no noise.
# Need to use a realisitic point where current is flowing to estimate the noise in current measurement.

STEP_FORWARD = 10
SAMPLING_RANGE = 500

sd_zero_current_array = []

for index in before_open['index']:
    start_index = min(len(compiled_df) - 1, index + STEP_FORWARD)
    end_index = min(len(compiled_df) - 1, start_index + SAMPLING_RANGE)
    mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
    mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
    sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
    sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
    
    mean_current = (mean_current_1 + mean_current_2) / 2
    sd_current = (sd_current_1 + sd_current_2) / 2
    sd_zero_current_array.append(sd_current)
    
    print(f"Mean Current: {mean_current:.4f} A, SD Current: {sd_current:.4f} A")
    
sigma_i = np.mean(sd_zero_current_array)
print(f"Mean Standard Deviations: {sigma_i:.4f} A")

Mean Current: 0.0238 A, SD Current: 0.0083 A
Mean Current: 0.0381 A, SD Current: 0.0076 A
Mean Current: 0.0099 A, SD Current: 0.0086 A
Mean Current: 0.0038 A, SD Current: 0.0086 A
Mean Current: -0.0092 A, SD Current: 0.0082 A
Mean Current: 0.0005 A, SD Current: 0.0084 A
Mean Current: 0.0031 A, SD Current: 0.0081 A
Mean Current: -0.0004 A, SD Current: 0.0084 A
Mean Current: -0.0179 A, SD Current: 0.0086 A
Mean Current: 0.0017 A, SD Current: 0.0080 A
Mean Current: -0.0098 A, SD Current: 0.0084 A
Mean Current: 0.0032 A, SD Current: 0.0080 A
Mean Current: 0.0043 A, SD Current: 0.0077 A
Mean Standard Deviations: 0.0082 A


In [8]:
STEP_BACK = 10
SAMPLING_RANGEs = [100, 200, 300, 400]

for sampling_range in SAMPLING_RANGEs:
    sd_current_array = []

    for index in before_open['index']:
        end_index = max(0, index - STEP_BACK)
        start_index = max(0, end_index - sampling_range)
        mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
        mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
        
        sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
        sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
        
        mean_current = (mean_current_1 + mean_current_2) / 2
        sd_current = (sd_current_1 + sd_current_2) / 2
        
        sd_current_array.append(sd_current)
        #print(f"Mean Current: {mean_current_1:.4f} A, SD Current: {sd_current_1:.4f} A")
    print(f"Mean Standard Deviations ({sampling_range}) samples: {np.mean(sd_current_array):.4f} A")

Mean Standard Deviations (100) samples: 0.0110 A
Mean Standard Deviations (200) samples: 0.0163 A
Mean Standard Deviations (300) samples: 0.0224 A
Mean Standard Deviations (400) samples: 0.0294 A


In [9]:
STEP_BACK = 10
SAMPLING_RANGE = 200

sd_current_array = []

for index in before_open['index']:
    end_index = max(0, index - STEP_BACK)
    start_index = max(0, end_index - SAMPLING_RANGE)
    mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
    mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
    
    sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
    sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
    
    mean_current = (mean_current_1 + mean_current_2) / 2
    sd_current = (sd_current_1 + sd_current_2) / 2
    
    sd_current_array.append(sd_current)
    print(f"Mean Current: {mean_current_1:.4f} A, SD Current: {sd_current:.4f} A")
    
sigma_i = np.mean(sd_current_array)
print(f"Mean Standard Deviations: {sigma_i:.4f} A")

Mean Current: 27.8306 A, SD Current: 0.0208 A
Mean Current: 26.6799 A, SD Current: 0.0137 A
Mean Current: 26.4745 A, SD Current: 0.0155 A
Mean Current: 26.8020 A, SD Current: 0.0224 A
Mean Current: 25.9826 A, SD Current: 0.0149 A
Mean Current: 25.6800 A, SD Current: 0.0175 A
Mean Current: 25.2777 A, SD Current: 0.0144 A
Mean Current: 24.9651 A, SD Current: 0.0099 A
Mean Current: 26.0286 A, SD Current: 0.0144 A
Mean Current: 25.1963 A, SD Current: 0.0142 A
Mean Current: 25.1943 A, SD Current: 0.0171 A
Mean Current: 25.0988 A, SD Current: 0.0174 A
Mean Current: 23.9293 A, SD Current: 0.0203 A
Mean Standard Deviations: 0.0163 A


In [10]:
sigma_kcl = np.sqrt(4* sigma_i**2)
print(f"Sigma KCL: {sigma_kcl:.4f} A")

Sigma KCL: 0.0327 A


In [13]:
battery_state = pd.read_csv(battery_state_file_name)
soc_factor = len(str(battery_state['SoC'].iloc[1]).split('.')[1])

print(soc_factor)

3


In [14]:
battery_constant = {
    'Q_total': total_Ah,
    'sigma_v': sigma_v,
    'sigma_i': sigma_i,
    'sigma_kcl': sigma_kcl,
    'interval_factor': soc_factor
}

print(f"Battery Constants: {json.dumps(battery_constant, indent=4)}")

Battery Constants: {
    "Q_total": 73.36907721954144,
    "sigma_v": 0.0018569382671418761,
    "sigma_i": 0.01634680930254865,
    "sigma_kcl": 0.0326936186050973,
    "interval_factor": 3
}


In [15]:
json.dump(battery_constant, open(output_file_name, 'w'), indent=4)
print(f"Battery constants saved to {output_file_name}")

Battery constants saved to c:\Users\Kor\Documents\GitHub\Proa-II-Electrical-Setup\Calibration Data 2\Data\Completed Discharge LiNMC data\battery_constants.json
